<a href="https://colab.research.google.com/github/futuregunmulju/2026-Signals-and-Systems-Speech-Noise-Reduction/blob/main/%5B9%EC%A1%B0%5D_Challenge1_Speech_Noise_Reduction_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Band-pass -> MMSE-STSA -> MMSE-LSA

In [1]:
%pip install -q "numpy>=1.26,<3" "scipy>=1.12,<2" "soundfile>=0.12,<1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.special import i0, i1, exp1


INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"입력 폴더: {INPUT_DIR.resolve()}")
print(f"출력 폴더: {OUTPUT_DIR.resolve()}")
print("input 폴더에 테스트 WAV를 넣은 뒤 나머지 셀을 실행하세요.")

In [4]:
SR = 16000
N_FFT = 512
HOP = 128
NOISE_FRACTION = 0.20
SMOOTHING = 0.98
GAIN_FLOOR = 0.05


def stft_audio(audio):
    if len(audio) < N_FFT:
        audio = np.pad(audio, (0, N_FFT - len(audio)))

    return signal.stft(
        audio,
        fs=SR,
        window="hann",
        nperseg=N_FFT,
        noverlap=N_FFT - HOP,
        nfft=N_FFT,
        boundary="zeros",
        padded=True,
    )


def inverse_stft(spectrum, length):
    _, audio = signal.istft(
        spectrum,
        fs=SR,
        window="hann",
        nperseg=N_FFT,
        noverlap=N_FFT - HOP,
        nfft=N_FFT,
        input_onesided=True,
        boundary=True,
    )

    if len(audio) < length or not np.isfinite(audio).all():
        raise ValueError("음성 복원 결과가 유효하지 않습니다.")

    return audio[:length]


def estimate_noise_power(spectrum, times, length):
    power = np.abs(spectrum) ** 2
    centers = np.rint(times * SR).astype(int)

    complete = np.flatnonzero(
        (centers >= N_FFT // 2)
        & (centers + N_FFT // 2 <= length)
    )
    if len(complete) == 0:
        complete = np.arange(power.shape[1])

    energy = power[:, complete].mean(axis=0)
    count = max(1, int(np.ceil(NOISE_FRACTION * len(complete))))
    selected = complete[np.argsort(energy, kind="stable")[:count]]
    noise = power[:, selected].mean(axis=1)

    return np.maximum(
        uniform_filter1d(noise, size=3, mode="nearest"),
        1e-18,
    )


def estimate_snr(spectrum, noise):
    power = np.abs(spectrum) ** 2
    gamma = np.maximum(power / noise, 1e-12)
    xi = np.maximum(gamma - 1.0, 0.0)

    for frame in range(1, spectrum.shape[1]):
        xi[:, frame] = (
            SMOOTHING
            * (np.abs(spectrum[:, frame - 1]) ** 2 / noise[:, 0])
            + (1.0 - SMOOTHING)
            * np.maximum(gamma[:, frame] - 1.0, 0.0)
        )

    v = np.maximum(gamma * xi / (1.0 + xi), 1e-12)
    return gamma, xi, v


def band_pass_filter(noisy):
    sos = signal.butter(
        4,
        (120.0, 3800.0),
        btype="bandpass",
        fs=SR,
        output="sos",
    )
    padlen = min(3 * (2 * len(sos) + 1), len(noisy) - 1)
    return signal.sosfiltfilt(sos, noisy, padlen=padlen)


def mmse_stsa_filter(noisy):
    _, times, spectrum = stft_audio(noisy)
    noise = estimate_noise_power(
        spectrum, times, len(noisy)
    )[:, None]

    gamma, xi, v = estimate_snr(spectrum, noise)
    half_v = np.minimum(v / 2.0, 700.0)

    # 기존 구현의 계산식과 비유한 값 처리 유지
    with np.errstate(over="ignore", invalid="ignore"):
        gain = (
            np.sqrt(np.pi * v) / (2.0 * gamma)
            * np.exp(-half_v)
            * ((1.0 + v) * i0(half_v) + v * i1(half_v))
        )

    gain = np.clip(
        np.nan_to_num(gain, nan=0.0, posinf=1.0),
        GAIN_FLOOR,
        1.0,
    )
    return inverse_stft(gain * spectrum, len(noisy))


def mmse_lsa_filter(noisy):
    _, times, spectrum = stft_audio(noisy)
    noise = estimate_noise_power(
        spectrum, times, len(noisy)
    )[:, None]

    gamma, xi, v = estimate_snr(spectrum, noise)
    gain = (xi / (1.0 + xi)) * np.exp(0.5 * exp1(v))
    gain = np.clip(
        np.nan_to_num(gain, nan=0.0, posinf=1.0),
        GAIN_FLOOR,
        1.0,
    )
    return inverse_stft(gain * spectrum, len(noisy))


def denoise(noisy):
    noisy = np.asarray(noisy, dtype=np.float64)

    if noisy.ndim != 1 or noisy.size == 0:
        raise ValueError("비어 있지 않은 단일 채널 파형이 필요합니다.")
    if not np.isfinite(noisy).all():
        raise ValueError("입력에 유효하지 않은 값이 있습니다.")

    enhanced = band_pass_filter(noisy)
    enhanced = mmse_stsa_filter(enhanced)
    enhanced = mmse_lsa_filter(enhanced)
    return enhanced

In [5]:
wav_files = sorted(
    path
    for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".wav"
)

if not wav_files:
    raise FileNotFoundError(
        f"테스트 WAV를 먼저 넣어주세요: {INPUT_DIR.resolve()}"
    )

failures = []
success_count = 0

for index, input_path in enumerate(wav_files, start=1):
    output_path = OUTPUT_DIR / input_path.name
    temporary_path = OUTPUT_DIR / f".{input_path.name}.tmp"

    try:
        # 항상 (샘플 수, 채널 수) 형태로 읽음
        noisy, sample_rate = sf.read(
            input_path,
            dtype="float64",
            always_2d=True,
        )

        if sample_rate != SR:
            raise ValueError(
                f"16 kHz 입력이 필요합니다. 현재 {sample_rate} Hz"
            )
        if len(noisy) == 0 or not np.isfinite(noisy).all():
            raise ValueError("입력 음성이 비어 있거나 유효하지 않습니다.")

        enhanced = np.empty_like(noisy)

        for channel in range(noisy.shape[1]):
            enhanced[:, channel] = denoise(noisy[:, channel])

        if enhanced.shape != noisy.shape:
            raise ValueError("처리 후 샘플 수 또는 채널 수가 달라졌습니다.")
        if not np.isfinite(enhanced).all():
            raise ValueError("출력에 유효하지 않은 값이 있습니다.")

        sf.write(
            temporary_path,
            enhanced,
            sample_rate,
            format="WAV",
            subtype="FLOAT",
        )

        # 저장된 파일의 형식 확인
        info = sf.info(temporary_path)
        if (
            info.samplerate != sample_rate
            or info.channels != noisy.shape[1]
            or info.frames != noisy.shape[0]
        ):
            raise ValueError("저장된 WAV의 형식이 입력과 일치하지 않습니다.")

        temporary_path.replace(output_path)
        success_count += 1
        print(f"[{index}/{len(wav_files)}] 저장 완료: {output_path.name}")

    except Exception as error:
        temporary_path.unlink(missing_ok=True)
        failures.append((input_path.name, str(error)))
        print(f"[{index}/{len(wav_files)}] 실패: {input_path.name} — {error}")

print(f"\n처리 완료: {success_count}/{len(wav_files)}개")
print(f"출력 폴더: {OUTPUT_DIR.resolve()}")

if failures:
    raise RuntimeError(
        f"{len(failures)}개 파일 처리에 실패했습니다. 위 오류를 확인하세요."
    )

Validation:   0%|          | 0/164 [00:00<?, ?it/s]

/tmp/ipykernel_3171/3626680540.py:100: RuntimeWarning: overflow encountered in multiply
  * ((1.0 + v) * i0(half_v) + v * i1(half_v))
/tmp/ipykernel_3171/3626680540.py:100: RuntimeWarning: overflow encountered in add
  * ((1.0 + v) * i0(half_v) + v * i1(half_v))



Band-pass → MMSE-STSA → MMSE-LSA
Validation 164쌍 평균 WB-PESQ: 1.612048
